In [1]:
import os
os.chdir(r'C:\Users\adhil\OneDrive\Desktop\Phishing Project\phishing_extension')
print("✅ Current folder:", os.getcwd())

✅ Current folder: C:\Users\adhil\OneDrive\Desktop\Phishing Project\phishing_extension


In [2]:
import os
os.makedirs('icons', exist_ok=True)
print("✅ icons folder created!")

✅ icons folder created!


In [3]:
%%writefile manifest.json
{
  "manifest_version": 3,
  "name": "PhishGuard - Phishing Detector",
  "version": "1.0.0",
  "description": "Detects phishing websites in real-time using Machine Learning.",
  "action": {
    "default_popup": "popup.html"
  },
  "permissions": [
    "activeTab",
    "storage",
    "tabs"
  ],
  "host_permissions": [
    "http://localhost:5000/*"
  ],
  "background": {
    "service_worker": "background.js"
  },
  "content_scripts": [
    {
      "matches": ["<all_urls>"],
      "js": ["content.js"],
      "run_at": "document_idle"
    }
  ]
}

Writing manifest.json


In [4]:
%%writefile popup.html
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>PhishGuard</title>
  <style>
    * { box-sizing: border-box; margin: 0; padding: 0; }
    body {
      width: 340px;
      font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
      background: #0f1117;
      color: #e2e8f0;
      font-size: 14px;
    }
    .header {
      background: #1a1f2e;
      padding: 16px 18px;
      border-bottom: 1px solid #1e2535;
      display: flex;
      align-items: center;
      gap: 10px;
    }
    .logo {
      width: 32px; height: 32px;
      background: #2563eb;
      border-radius: 8px;
      display: flex; align-items: center; justify-content: center;
      font-size: 18px;
    }
    .header-text h1 { font-size: 15px; font-weight: 600; color: #f1f5f9; }
    .header-text p  { font-size: 11px; color: #64748b; }
    .url-bar {
      padding: 12px 18px;
      border-bottom: 1px solid #1e2535;
    }
    .url-label { font-size: 10px; color: #64748b; text-transform: uppercase; margin-bottom: 5px; }
    .url-text  { font-size: 12px; color: #94a3b8; word-break: break-all; max-height: 48px; overflow: hidden; }
    .result-card {
      margin: 14px 18px;
      border-radius: 12px;
      padding: 16px;
    }
    .result-card.loading { background: #1e2535; border: 1px solid #2d3748; }
    .result-card.safe    { background: #052e16; border: 1px solid #166534; }
    .result-card.danger  { background: #2d0a0a; border: 1px solid #7f1d1d; }
    .result-card.warning { background: #1c1506; border: 1px solid #713f12; }
    .result-icon  { font-size: 32px; text-align: center; margin-bottom: 10px; }
    .result-title { font-size: 16px; font-weight: 700; text-align: center; margin-bottom: 4px; }
    .safe    .result-title { color: #4ade80; }
    .danger  .result-title { color: #f87171; }
    .warning .result-title { color: #fbbf24; }
    .loading .result-title { color: #94a3b8; }
    .result-sub { font-size: 12px; text-align: center; color: #64748b; }
    .conf-section { margin: 0 18px 12px; }
    .conf-label   { display: flex; justify-content: space-between; font-size: 11px; color: #64748b; margin-bottom: 5px; }
    .conf-bar-bg  { height: 6px; background: #1e2535; border-radius: 3px; overflow: hidden; }
    .conf-bar-fill { height: 100%; border-radius: 3px; transition: width .6s ease; }
    .safe-fill    { background: #22c55e; }
    .danger-fill  { background: #ef4444; }
    .warning-fill { background: #f59e0b; }
    .flags-section { margin: 0 18px 12px; }
    .flags-title   { font-size: 11px; color: #64748b; text-transform: uppercase; margin-bottom: 7px; }
    .flag-item {
      display: flex; align-items: center; gap: 7px;
      font-size: 12px; color: #fca5a5;
      padding: 5px 8px; background: #2d0a0a;
      border-radius: 6px; margin-bottom: 4px;
    }
    .flag-dot { width: 5px; height: 5px; background: #ef4444; border-radius: 50%; flex-shrink: 0; }
    .btn-section { padding: 0 18px 14px; }
    .btn-check {
      width: 100%; padding: 9px;
      background: #2563eb; color: #fff;
      border: none; border-radius: 8px;
      font-size: 13px; font-weight: 500;
      cursor: pointer;
    }
    .btn-check:hover    { background: #1d4ed8; }
    .btn-check:disabled { background: #1e2535; color: #4a5568; cursor: not-allowed; }
    .custom-section { padding: 0 18px 14px; border-top: 1px solid #1e2535; }
    .custom-label   { font-size: 11px; color: #64748b; text-transform: uppercase; margin: 10px 0 6px; }
    .custom-row     { display: flex; gap: 6px; }
    .custom-input {
      flex: 1; padding: 8px 10px;
      background: #1e2535; border: 1px solid #2d3748;
      border-radius: 7px; color: #e2e8f0; font-size: 12px; outline: none;
    }
    .custom-input:focus { border-color: #2563eb; }
    .btn-scan {
      padding: 8px 12px; background: #1e2535;
      border: 1px solid #2d3748; border-radius: 7px;
      color: #94a3b8; font-size: 12px; cursor: pointer;
    }
    .btn-scan:hover { background: #2563eb; color: #fff; border-color: #2563eb; }
    .footer {
      padding: 8px 18px; border-top: 1px solid #1e2535;
      display: flex; justify-content: space-between; align-items: center;
    }
    .footer-status { font-size: 10px; color: #4a5568; display: flex; align-items: center; gap: 5px; }
    .status-dot    { width: 6px; height: 6px; border-radius: 50%; }
    .status-dot.online  { background: #22c55e; }
    .status-dot.offline { background: #ef4444; }
    .footer-method { font-size: 10px; color: #4a5568; }
    .spinner {
      display: block; width: 16px; height: 16px;
      border: 2px solid #2d3748; border-top-color: #2563eb;
      border-radius: 50%; animation: spin .7s linear infinite; margin: 0 auto 10px;
    }
    @keyframes spin { to { transform: rotate(360deg); } }
  </style>
</head>
<body>
  <div class="header">
    <div class="logo">🛡</div>
    <div class="header-text">
      <h1>PhishGuard</h1>
      <p>ML-powered phishing detector</p>
    </div>
  </div>
  <div class="url-bar">
    <div class="url-label">Current page</div>
    <div class="url-text" id="currentUrl">Loading...</div>
  </div>
  <div class="result-card loading" id="resultCard">
    <div class="result-icon">🔍</div>
    <div class="result-title" id="resultTitle">Not checked</div>
    <div class="result-sub" id="resultSub">Click "Check This Page" to analyze</div>
  </div>
  <div class="conf-section" id="confSection" style="display:none">
    <div class="conf-label">
      <span>Confidence</span>
      <span id="confValue">0%</span>
    </div>
    <div class="conf-bar-bg">
      <div class="conf-bar-fill" id="confFill" style="width:0%"></div>
    </div>
  </div>
  <div class="flags-section" id="flagsSection" style="display:none">
    <div class="flags-title">Detection flags</div>
    <div id="flagsList"></div>
  </div>
  <div class="btn-section">
    <button class="btn-check" id="checkBtn">🔍 Check This Page</button>
  </div>
  <div class="custom-section">
    <div class="custom-label">Scan any URL</div>
    <div class="custom-row">
      <input type="text" class="custom-input" id="customUrl" placeholder="https://example.com">
      <button class="btn-scan" id="scanBtn">Scan</button>
    </div>
  </div>
  <div class="footer">
    <div class="footer-status">
      <div class="status-dot offline" id="statusDot"></div>
      <span id="statusText">Checking API...</span>
    </div>
    <div class="footer-method" id="methodText"></div>
  </div>
  <script src="popup.js"></script>
</body>
</html>

Writing popup.html


In [5]:
%%writefile popup.js
const API_BASE = "http://localhost:5000";

async function checkApiHealth() {
  try {
    const res = await fetch(`${API_BASE}/health`, { signal: AbortSignal.timeout(2500) });
    const data = await res.json();
    return data.status === "ok";
  } catch { return false; }
}

async function predictUrl(url) {
  const res = await fetch(`${API_BASE}/predict`, {
    method: "POST",
    headers: { "Content-Type": "application/json" },
    body: JSON.stringify({ url }),
    signal: AbortSignal.timeout(6000),
  });
  if (!res.ok) throw new Error(`API error ${res.status}`);
  return res.json();
}

function getCurrentTabUrl() {
  return new Promise(resolve => {
    chrome.tabs.query({ active: true, currentWindow: true }, tabs => {
      resolve(tabs[0]?.url || "");
    });
  });
}

function showLoading() {
  const card = document.getElementById("resultCard");
  card.className = "result-card loading";
  card.innerHTML = `
    <div class="spinner"></div>
    <div class="result-title" style="color:#94a3b8">Analyzing...</div>
    <div class="result-sub">Contacting detection API</div>`;
  document.getElementById("confSection").style.display  = "none";
  document.getElementById("flagsSection").style.display = "none";
}

function showResult(data) {
  const card       = document.getElementById("resultCard");
  const isPhishing = data.is_phishing;
  const conf       = Math.round(data.confidence);
  const flags      = data.flags || [];

  if (isPhishing) {
    card.className = "result-card danger";
    card.innerHTML = `
      <div class="result-icon">⚠️</div>
      <div class="result-title">Phishing Detected!</div>
      <div class="result-sub">This site may be trying to steal your data</div>`;
  } else if (conf < 70) {
    card.className = "result-card warning";
    card.innerHTML = `
      <div class="result-icon">⚠️</div>
      <div class="result-title">Suspicious</div>
      <div class="result-sub">Proceed with caution</div>`;
  } else {
    card.className = "result-card safe";
    card.innerHTML = `
      <div class="result-icon">✅</div>
      <div class="result-title">Looks Safe</div>
      <div class="result-sub">No phishing indicators found</div>`;
  }

  const confSection = document.getElementById("confSection");
  const fillClass   = isPhishing ? "danger-fill" : conf < 70 ? "warning-fill" : "safe-fill";
  confSection.style.display = "block";
  document.getElementById("confValue").textContent = `${conf}%`;
  const fill = document.getElementById("confFill");
  fill.className = `conf-bar-fill ${fillClass}`;
  setTimeout(() => { fill.style.width = `${conf}%`; }, 50);

  if (flags.length > 0) {
    const flagsSection = document.getElementById("flagsSection");
    const flagsList    = document.getElementById("flagsList");
    flagsSection.style.display = "block";
    flagsList.innerHTML = flags.map(f =>
      `<div class="flag-item"><div class="flag-dot"></div>${f}</div>`
    ).join('');
  }
  document.getElementById("methodText").textContent = data.method || "";
}

function showError(msg) {
  const card = document.getElementById("resultCard");
  card.className = "result-card loading";
  card.innerHTML = `
    <div class="result-icon">❌</div>
    <div class="result-title" style="color:#f87171">Error</div>
    <div class="result-sub" style="color:#94a3b8">${msg}</div>`;
}

function updateApiStatus(online) {
  const dot  = document.getElementById("statusDot");
  const text = document.getElementById("statusText");
  if (online) {
    dot.className    = "status-dot online";
    text.textContent = "API connected";
  } else {
    dot.className    = "status-dot offline";
    text.textContent = "API offline - start Flask";
  }
}

async function init() {
  const url   = await getCurrentTabUrl();
  const urlEl = document.getElementById("currentUrl");
  urlEl.textContent = url || "Unknown";

  const online = await checkApiHealth();
  updateApiStatus(online);

  document.getElementById("checkBtn").addEventListener("click", async () => {
    const btn = document.getElementById("checkBtn");
    btn.disabled    = true;
    btn.textContent = "Checking...";
    showLoading();
    try {
      const result = await predictUrl(url);
      showResult(result);
    } catch (e) {
      showError("Could not reach API. Is Flask running?");
    } finally {
      btn.disabled    = false;
      btn.textContent = "🔍 Check This Page";
    }
  });

  document.getElementById("scanBtn").addEventListener("click", async () => {
    const input   = document.getElementById("customUrl");
    const scanUrl = input.value.trim();
    if (!scanUrl) return;
    const fullUrl = scanUrl.startsWith("http") ? scanUrl : `https://${scanUrl}`;
    document.getElementById("currentUrl").textContent = fullUrl;
    showLoading();
    try {
      const result = await predictUrl(fullUrl);
      showResult(result);
    } catch (e) {
      showError("Could not reach API. Is Flask running?");
    }
  });

  document.getElementById("customUrl").addEventListener("keydown", e => {
    if (e.key === "Enter") document.getElementById("scanBtn").click();
  });

  if (online && url && (url.startsWith("http://") || url.startsWith("https://"))) {
    document.getElementById("checkBtn").click();
  }
}

init();

Writing popup.js


In [6]:
%%writefile background.js
const API_BASE = "http://localhost:5000";
const cache    = {};

async function checkUrl(url) {
  if (cache[url]) return cache[url];
  try {
    const res  = await fetch(`${API_BASE}/predict`, {
      method:  "POST",
      headers: { "Content-Type": "application/json" },
      body:    JSON.stringify({ url }),
      signal:  AbortSignal.timeout(5000),
    });
    if (!res.ok) return null;
    const data = cache[url] = await res.json();
    return data;
  } catch { return null; }
}

chrome.tabs.onUpdated.addListener(async (tabId, changeInfo, tab) => {
  if (changeInfo.status !== "complete" || !tab.url) return;
  if (!tab.url.startsWith("http")) return;

  const result = await checkUrl(tab.url);
  if (!result) return;

  if (result.is_phishing) {
    chrome.action.setBadgeText({ tabId, text: "!" });
    chrome.action.setBadgeBackgroundColor({ tabId, color: "#ef4444" });
  } else {
    chrome.action.setBadgeText({ tabId, text: "" });
  }
});

Writing background.js


In [7]:
%%writefile content.js
const API_BASE = "http://localhost:5000";

(async () => {
  const url = window.location.href;
  if (!url.startsWith("http")) return;

  try {
    const res = await fetch(`${API_BASE}/predict`, {
      method:  "POST",
      headers: { "Content-Type": "application/json" },
      body:    JSON.stringify({ url }),
      signal:  AbortSignal.timeout(4000),
    });
    if (!res.ok) return;
    const data = await res.json();
    if (data.is_phishing) showWarningBanner(data);
  } catch { }
})();

function showWarningBanner(data) {
  if (document.getElementById("phishguard-banner")) return;
  const banner = document.createElement("div");
  banner.id = "phishguard-banner";
  banner.style.cssText = `
    position: fixed; top: 0; left: 0; right: 0;
    z-index: 2147483647; background: #7f1d1d;
    color: #fecaca; font-family: sans-serif; font-size: 14px;
    padding: 12px 16px; display: flex; align-items: center; gap: 12px;
    box-shadow: 0 2px 12px rgba(0,0,0,0.4); border-bottom: 2px solid #ef4444;`;
  const conf  = Math.round(data.confidence);
  const flags = (data.flags || []).slice(0, 3).join(" · ");
  banner.innerHTML = `
    <span style="font-size:20px">⚠️</span>
    <div style="flex:1">
      <strong style="color:#f87171;font-size:15px;">PhishGuard Warning: Potential Phishing Site</strong>
      <div style="margin-top:2px;font-size:12px;opacity:.9;">
        Confidence: ${conf}%${flags ? ` · ${flags}` : ""}
      </div>
    </div>
    <button id="phishguard-dismiss" style="
      background:transparent; border:1px solid #ef4444; color:#fca5a5;
      padding:5px 12px; border-radius:6px; cursor:pointer; font-size:12px;">
      Dismiss
    </button>`;
  document.body.prepend(banner);
  document.getElementById("phishguard-dismiss").addEventListener("click", () => banner.remove());
  setTimeout(() => banner.remove(), 15000);
}

Writing content.js


In [8]:
import os

path  = r'C:\Users\adhil\OneDrive\Desktop\Phishing Project\phishing_extension'
files = os.listdir(path)

print("Files in phishing_extension:")
for f in files:
    print(" ✅", f)

Files in phishing_extension:
 ✅ .ipynb_checkpoints
 ✅ background.js
 ✅ content.js
 ✅ icons
 ✅ manifest.json
 ✅ popup.html
 ✅ popup.js
 ✅ setup_extension.ipynb
